# Test 02: Supervised and Unsupervised Learning

Name: Lon Cherryholmes, Sr.

Date: Friday 12/05/2025

CWID: xxx17840

You need to get the questions from our class MyLeoOnline site by starting Test 02.  You should be given 2 multi-part questions.

Please make sure that the notebook that you submit for grading runs all cells cleanly from top to bottom if a **Run -> Run All Cells** is performed.  Please give your name and CWID in the provided fields below to ensure your test work is properly attributed.  As a reminder, all work on tests and assignments for this class is individual work unless otherwise noted.  You may not work with others, nor use past student's works or answers created by others for these test questions.

## General Setup

The following imports may be useful for Test 02, and should be sufficient to finish the
questions.  However, if you would like to import additional functions or objects, you may
do so, though they need to come from `scikit-learn` library, or other functions from basic
libraries like `numpy`, `matplotlib` or `pandas`.

In [ ]:
# Basic imports that are generally useful
import numpy as np
import pandas as pd
import seaborn as sbn
import matplotlib.pyplot as plt
import matplotlib
import sklearn

# Imports specific to these test questions
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import confusion_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Please put all imports you add for your work here
from google.colab import drive
from pprint import pprint, pformat
from pandas.plotting import scatter_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Notebook-wide options to improve appearance of basic matplotlib figures and numpy output
%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (10, 8) # set default figure size, 10in by 8in
np.set_printoptions(suppress=True)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)
plt.rc('figure', titlesize=18)
plt.rc('legend', fontsize=14)
plt.rcParams['figure.figsize'] = (12.0, 8.0) # default figure size if not specified in plot
# Please put all options you add for your work here
pd.set_option("display.max_colwidth", 20)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
np.set_printoptions(linewidth=120, precision=3)

Start Test 02 in MyLeoOnline to get the test questions you need to answer.  You will need
to use the `wine` dataset.  In addition to this notebook, you should have been given
a `wine.csv` file to download for use on this question.  Please put the file
into the same directory as where you run your notebook and do not use an absolute
path name in your command to load the file into a Pandas dataframe or NumPy array.

In [ ]:
drive.mount('/content/modules', force_remount=False)
wine = pd.read_csv("/content/modules/My Drive/CSCI-574/wine.csv", delimiter=';')

In [ ]:
def quality_rank(q):
    if q >= 7:    return 1 # premium
    elif q >= 5:  return 2 # good
    else:         return 3 # poor
wine["rank"] = wine["quality"].apply(quality_rank)

In [ ]:
loaded_wine = load_wine()
print(loaded_wine.data.shape)
print(loaded_wine.feature_names)
print(loaded_wine.frame)
print(loaded_wine.target.shape)
print(loaded_wine.target_names)
print(loaded_wine.DESCR)

## Data Exploration

In [ ]:
print(wine.info())

In [ ]:
wine['quality'].value_counts()

In [ ]:
wine['rank'].value_counts()

In [ ]:
display(wine.describe())

In [ ]:
wine.hist(bins=64, figsize=(12,8));

In [ ]:
wine_corr = wine.corr()
print(wine_corr['quality'].sort_values(ascending=False))

In [ ]:
scatter_matrix(wine[['quality', 'rank', 'alcohol', 'density', 'chlorides', 'volatile acidity', 'total sulfur dioxide']], figsize=(14,10));

## Prep 1: Supervised Learning

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
strat_splits = []
for train_index, test_index in splitter.split(wine, wine['rank']):
    strat_splits.append([wine.iloc[train_index], wine.iloc[test_index]])

In [ ]:
print(wine['rank'].value_counts() / len(wine))
pprint([(w['rank'].value_counts() / len(w)).to_numpy() for w, _ in strat_splits])

In [ ]:
wine_train, wine_test = strat_splits[0]
print(wine_train.shape)
print(wine_test.shape)
X_train = wine_train.iloc[:, 0:11].to_numpy()
y_train = wine_train.loc[:, 'rank'].to_numpy()
X_test  = wine_test.iloc[:, 0:11].to_numpy()
y_test  = wine_test.loc[:, 'rank'].to_numpy()
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
transformer = StandardScaler()
X_train_scaled = transformer.fit_transform(X_train)
X_test_scaled = transformer.fit_transform(X_test)
model = LogisticRegression(penalty='l2')
model.fit(X_train_scaled, y_train)
y_test_pred = model.predict(X_test_scaled)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred);

In [ ]:
transformer = StandardScaler()
X_train_scaled = transformer.fit_transform(X_train)
X_test_scaled = transformer.fit_transform(X_test)
model = SVC()
model.fit(X_train_scaled, y_train)
y_test_pred = model.predict(X_test_scaled)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred);

In [ ]:
model = ExtraTreesClassifier()
model.fit(X_train, y_train)
y_test_pred = model.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred);

In [ ]:
pipe = Pipeline([
    ('pre', StandardScaler()),
    ('clf', LogisticRegression())
])
pipe.fit(X_train, y_train)
y_test_pred = pipe.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred)
grid_params = [
    { 'clf__solver': ['lbfgs', 'newton-cholesky', 'newton-cg'],
      'clf__penalty': [None, 'l2'],
      'clf__max_iter': [512],
      'clf__class_weight': [None, 'balanced']
    },
    { 'clf__solver': ['liblinear'],
      'clf__penalty': ['l1', 'l2'],
      'clf__max_iter': [512],
      'clf__class_weight': [None, 'balanced']
    },
    { 'clf__solver': ['saga'],
      'clf__penalty': [None, 'l1', 'l2'],
      'clf__max_iter': [2048],
      'clf__class_weight': [None, 'balanced']
    },
    { 'clf__solver': ['saga'],
      'clf__penalty': ['elasticnet'],
      'clf__max_iter': [2048],
      'clf__l1_ratio': [0.0, 0.25, 0.5, 0.75, 1.0],
      'clf__class_weight': [None, 'balanced']
    },
]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipe, grid_params, cv=cv, scoring='accuracy')
grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
X = wine.iloc[:, 0:11].to_numpy()
y = wine.loc[:, 'rank'].to_numpy()
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_indices, test_indices = next(splitter.split(X, y))
X_train, y_train = X[train_indices], y[train_indices]
X_test, y_test = X[test_indices], y[test_indices]
model = ExtraTreesClassifier(n_estimators=128, random_state=42)
model.fit(X_train, y_train)
importances = model.feature_importances_
feature_names = wine.columns.tolist()
feature_importances = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)
for name, score in feature_importances:
    print(f"{name:>22s}: {score:.4f}")
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(model.score(X_test, y_test))

In [ ]:
logistic_pipe = Pipeline([
    ("pre", StandardScaler()),
    ("clf", LogisticRegression(solver="lbfgs", max_iter=512))
])
logistic_pipe.fit(X_train, y_train)
y_logistic_pred = logistic_pipe.predict(X_test)
print(confusion_matrix(y_test, y_logistic_pred))
print(logistic_pipe.score(X_test, y_test))
svc_pipe = Pipeline([
    ("pre", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=1.0))
])
svc_pipe.fit(X_train, y_train)
y_svc_pred = svc_pipe.predict(X_test)
print(confusion_matrix(y_test, y_svc_pred))
print(svc_pipe.score(X_test, y_test))

## Question 1: Supervised Learning (65 pts)

In [ ]:
X = wine.iloc[:, 0:11].to_numpy()
y = wine.loc[:, 'rank'].to_numpy()
print(X.shape)
print(y.shape)

## Prep 2: Unsupervised Learning

In [ ]:
transformer = StandardScaler()
X = wine.iloc[:, 0:11].to_numpy()
y = wine.loc[:, 'rank'].to_numpy()
X_scaled = transformer.fit_transform(X)
print(X_scaled.shape)

In [ ]:
pca = PCA()
pca.fit(X_scaled)
print(pca.n_components_)
print(pca.components_)
print(pca.explained_variance_)
print(pca.explained_variance_ratio_)

In [ ]:
pca = PCA(n_components=0.75)
Xd = pca.fit_transform(X_scaled)
print(pca.components_)
print(pca.explained_variance_)
print(pca.explained_variance_ratio_)

In [ ]:
k = 7
kmeans = KMeans(n_clusters=k, n_init=3)
y_pred = kmeans.fit_predict(X_scaled)
print(y)
print(y_pred)
print(kmeans.cluster_centers_)

In [ ]:
transformer = StandardScaler()
X_scaled = transformer.fit_transform(X)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="viridis", alpha=0.25)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Wine PCA")
plt.colorbar(label="Wine Quality")
plt.show()

In [ ]:
transformer = StandardScaler()
X_scaled = transformer.fit_transform(X)
pca = PCA(random_state=42)
pca.fit(X_scaled)
lngth = len(pca.explained_variance_ratio_)
plt.plot(range(1, lngth + 1), pca.explained_variance_ratio_, marker='o')
plt.plot(range(1, lngth + 1), np.cumsum(pca.explained_variance_ratio_), marker='x')
plt.xlabel("Principal Component")
plt.ylabel("Variance Explained Ratio")
plt.title("PCA Components")
plt.xticks(range(1, lngth + 1))
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
loadings = pd.DataFrame(pca.components_[0:6].T, columns=["PC1","PC2","PC3","PC4","PC5","PC6"], index=wine.columns[0:11])
print(loadings)

In [ ]:
transformer = StandardScaler()
X_scaled = transformer.fit_transform(X)
pca_xd = PCA(n_components=0.80, random_state=42)
X_pca_xd = pca_xd.fit_transform(X_scaled)
print(pca_xd.n_components_)
print(X_pca_xd.shape[1])
print(pca_xd.explained_variance_ratio_.sum())
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
clusters = kmeans.fit_predict(X_pca_xd)
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)
plt.subplot(1, 2, 1)
plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap="viridis", alpha=0.25)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA by quality")
plt.subplot(1, 2, 2)
plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=clusters, cmap="Accent", alpha=0.25)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA by KMeans clusters")
plt.tight_layout()
plt.show()

## Question 2: Unsupervised Learning (35 pts)

In [ ]:
X = wine.iloc[:, 0:11].to_numpy()
y = wine.loc[:, 'rank'].to_numpy()
print(X.shape)
print(y.shape)